In [1]:
import pathlib
import pandas as pd
import random
import string
import re
import numpy as np
import tensorflow as tf
import tensorflow.data as tf_data
import tensorflow.strings as tf_strings
import json
import keras
from keras import layers
from collections import Counter
from keras.layers import TextVectorization

In [2]:
DATA_DIR = pathlib.Path("data")
FRENCH_FILE = DATA_DIR / "CCMatrix.fr-ta.fr"
TAMIL_FILE = DATA_DIR / "CCMatrix.fr-ta.ta"
MAX_PAIRS = 400_000
VOCAB_SIZE = 15000
SEQUENCE_LENGTH = 20
BATCH_SIZE = 128
EMBED_DIM = 256
LATENT_DIM = 2048
NUM_HEADS = 8
EPOCHS = 3
STRIP_CHARS = string.punctuation + "¿"
STRIP_CHARS = STRIP_CHARS.replace("[", "").replace("]", "")

In [3]:
def is_clean_utf8(text: str) -> bool:
    try:
        text.encode('utf-8').decode('utf-8')
        return True
    except Exception:
        return False

In [4]:
def load_corpus(french_path: pathlib.Path, tamil_path: pathlib.Path):
    """Load parallel corpora, add start/end tokens to Tamil sentences and return two lists."""
    with open(str(tamil_path), encoding="utf-8", errors="ignore") as f:
        tamil_lines = f.read().split("\n")
    with open(str(french_path), encoding="utf-8", errors="ignore") as f:
        french_lines = f.read().split("\n")

    tamil_sentences = ["[start] " + line + " [end]" for line in tamil_lines]
    return french_lines, tamil_sentences

In [5]:
def build_dataframe(french_lines, tamil_sentences, max_pairs=MAX_PAIRS):
    clean_french = []
    clean_tamil = []
    for f_line, t_line in zip(french_lines, tamil_sentences):
        if is_clean_utf8(f_line) and is_clean_utf8(t_line) and f_line.strip() and t_line.strip():
            clean_french.append(f_line.strip())
            clean_tamil.append(t_line.strip())

    df = pd.DataFrame({"french": clean_french, "tamil": clean_tamil})
    df = df.head(max_pairs).reset_index(drop=True)
    df = df.sample(frac=1).reset_index(drop=True)
    return df

In [6]:
def custom_standardization(input_string):
    lowercase = tf_strings.lower(input_string)
    return tf_strings.regex_replace(lowercase, "[%s]" % re.escape(STRIP_CHARS), "")


In [7]:
def build_vectorizers(train_french_texts, train_tamil_texts, vocab_size=VOCAB_SIZE,
                      sequence_length=SEQUENCE_LENGTH):
    fre_vectorization = TextVectorization(
        max_tokens=vocab_size,
        output_mode="int",
        output_sequence_length=sequence_length,
    )

    tam_vectorization = TextVectorization(
        max_tokens=vocab_size,
        output_mode="int",
        output_sequence_length=sequence_length + 1,
        standardize=custom_standardization
    )

    fre_vectorization.adapt(train_french_texts)
    tam_vectorization.adapt(train_tamil_texts)
    return fre_vectorization, tam_vectorization

In [8]:
def save_vectorizer_and_vocab(vectorization, config_path, vocab_path, remove_standardize=True):
    config = vectorization.get_config()
    if remove_standardize:
        config.pop('standardize', None)
    with open(config_path, 'w', encoding='utf-8') as f:
        json.dump(config, f)

    vocab = [str(word) for word in vectorization.get_vocabulary()]
    with open(vocab_path, 'w', encoding='utf-8') as f:
        json.dump(vocab, f, ensure_ascii=False)

In [9]:
def build_and_save_fre_vocab(train_french_texts, config, vocab_path, vocab_size=VOCAB_SIZE):
    def simple_tokenize(text):
        text = text.lower()
        text = re.sub(rf"[{re.escape(STRIP_CHARS)}]", "", text)
        return text.split()

    all_words = []
    for line in train_french_texts:
        if isinstance(line, str):
            all_words.extend(simple_tokenize(line))

    most_common_words = [word for word, _ in Counter(all_words).most_common(vocab_size)]
    fre_vocab = ["[PAD]", "[UNK]"] + most_common_words

    with open(vocab_path, 'w', encoding='utf-8') as f:
        json.dump(fre_vocab, f, ensure_ascii=False)

In [10]:
def format_dataset(eng, spa, fre_vectorization, tam_vectorization):
    eng = fre_vectorization(eng)
    spa = tam_vectorization(spa)
    return (
        {
            "encoder_inputs": eng,
            "decoder_inputs": spa[:, :-1],
        },
        spa[:, 1:],
    )

In [11]:
def make_dataset(pairs_df, fre_vectorization, tam_vectorization, batch_size=BATCH_SIZE):
    eng_texts = list(pairs_df['french'])
    spa_texts = list(pairs_df['tamil'])
    dataset = tf_data.Dataset.from_tensor_slices((eng_texts, spa_texts))
    dataset = dataset.batch(batch_size)

    # Use a small wrapper to call format_dataset with vectorizers
    def _map_fn(eng, spa):
        return format_dataset(eng, spa, fre_vectorization, tam_vectorization)

    dataset = dataset.map(_map_fn)
    return dataset.cache().shuffle(2048).prefetch(16)


In [12]:
class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        self.attention = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.dense_proj = keras.Sequential([
            layers.Dense(dense_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.supports_masking = True

    def call(self, inputs, mask=None):
        if mask is not None:
            padding_mask = tf.cast(mask[:, None, :], dtype="int32")
        else:
            padding_mask = None

        attention_output = self.attention(query=inputs, value=inputs, key=inputs, attention_mask=padding_mask)
        proj_input = self.layernorm_1(inputs + attention_output)
        proj_output = self.dense_proj(proj_input)
        return self.layernorm_2(proj_input + proj_output)

    def get_config(self):
        config = super().get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "dense_dim": self.dense_dim,
            "num_heads": self.num_heads,
        })
        return config

In [13]:
class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_embeddings = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.position_embeddings = layers.Embedding(input_dim=sequence_length, output_dim=embed_dim)
        self.sequence_length = sequence_length
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim

    def call(self, inputs):
        length = tf.shape(inputs)[-1]
        positions = tf.range(start=0, limit=length, delta=1)
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

    def compute_mask(self, inputs, mask=None):
        return tf.not_equal(inputs, 0)

    def get_config(self):
        config = super().get_config()
        config.update({
            "sequence_length": self.sequence_length,
            "vocab_size": self.vocab_size,
            "embed_dim": self.embed_dim,
        })
        return config

In [14]:
class TransformerDecoder(layers.Layer):
    def __init__(self, embed_dim, latent_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.latent_dim = latent_dim
        self.num_heads = num_heads
        self.attention_1 = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.attention_2 = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.dense_proj = keras.Sequential([
            layers.Dense(latent_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.layernorm_3 = layers.LayerNormalization()
        self.supports_masking = True

    def call(self, inputs, encoder_outputs, mask=None):
        causal_mask = self.get_causal_attention_mask(inputs)
        if mask is not None:
            padding_mask = tf.cast(mask[:, None, :], dtype="int32")
            padding_mask = tf.minimum(padding_mask, causal_mask)
        else:
            padding_mask = None

        attention_output_1 = self.attention_1(query=inputs, value=inputs, key=inputs, attention_mask=causal_mask)
        out_1 = self.layernorm_1(inputs + attention_output_1)

        attention_output_2 = self.attention_2(
            query=out_1, value=encoder_outputs, key=encoder_outputs, attention_mask=padding_mask
        )
        out_2 = self.layernorm_2(out_1 + attention_output_2)

        proj_output = self.dense_proj(out_2)
        return self.layernorm_3(out_2 + proj_output)

    def get_causal_attention_mask(self, inputs):
        input_shape = tf.shape(inputs)
        batch_size, sequence_length = input_shape[0], input_shape[1]
        i = tf.range(sequence_length)[:, None]
        j = tf.range(sequence_length)
        mask = tf.cast(i >= j, dtype="int32")
        mask = tf.reshape(mask, (1, sequence_length, sequence_length))
        return tf.tile(mask, [batch_size, 1, 1])

    def get_config(self):
        config = super().get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "latent_dim": self.latent_dim,
            "num_heads": self.num_heads,
        })
        return config

In [15]:
def build_transformer(vocab_size=VOCAB_SIZE, sequence_length=SEQUENCE_LENGTH,
                      embed_dim=EMBED_DIM, latent_dim=LATENT_DIM, num_heads=NUM_HEADS):
    # Encoder
    encoder_inputs = keras.Input(shape=(None,), dtype="int64", name="encoder_inputs")
    x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(encoder_inputs)
    encoder_outputs = TransformerEncoder(embed_dim, latent_dim, num_heads)(x)
    encoder = keras.Model(encoder_inputs, encoder_outputs)

    # Decoder
    decoder_inputs = keras.Input(shape=(None,), dtype="int64", name="decoder_inputs")
    encoded_seq_inputs = keras.Input(shape=(None, embed_dim), name="decoder_state_inputs")
    x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(decoder_inputs)
    x = TransformerDecoder(embed_dim, latent_dim, num_heads)(x, encoded_seq_inputs)
    decoder_outputs = layers.Dense(vocab_size, activation="softmax")(x)

    decoder = keras.Model([decoder_inputs, encoded_seq_inputs], decoder_outputs)

    decoder_outputs = decoder([decoder_inputs, encoder_outputs])
    transformer = keras.Model([encoder_inputs, decoder_inputs], decoder_outputs, name="transformer")
    return transformer

In [16]:
def build_index_lookup(vectorization):
    vocab = vectorization.get_vocabulary()
    return dict(zip(range(len(vocab)), vocab))


In [17]:
def decode_sentence(input_sentence, fre_vectorization, tam_vectorization, transformer, tam_index_lookup,
                    max_decoded_sentence_length=SEQUENCE_LENGTH):
    tokenized_input_sentence = fre_vectorization([input_sentence])
    decoded_sentence = "[start]"
    for i in range(max_decoded_sentence_length):
        tokenized_target_sentence = tam_vectorization([decoded_sentence])[:, :-1]
        predictions = transformer([tokenized_input_sentence, tokenized_target_sentence])
        sampled_token_index = tf.argmax(predictions[0, i, :]).numpy().item(0)
        sampled_token = tam_index_lookup.get(sampled_token_index, "[UNK]")
        decoded_sentence += " " + sampled_token
        if sampled_token == "[end]":
            break
    return decoded_sentence

In [18]:
french_lines, tamil_sentences = load_corpus(FRENCH_FILE, TAMIL_FILE)
df = build_dataframe(french_lines, tamil_sentences, max_pairs=MAX_PAIRS)
print(f"{len(df)} total pairs")
    # train/val/test spli
num_val_samples = int(0.15 * len(df))
num_train_samples = len(df) - 2 * num_val_samples
train_pairs = df[:num_train_samples]
val_pairs = df[num_train_samples: num_train_samples + num_val_samples]
test_pairs = df[num_train_samples + num_val_samples:]

print(f"{len(train_pairs)} training pairs")
print(f"{len(val_pairs)} validation pairs")
print(f"{len(test_pairs)} test pairs")

train_fre_texts = train_pairs['french'].tolist()
train_tam_texts = train_pairs['tamil'].tolist()

fre_vectorization, tam_vectorization = build_vectorizers(train_fre_texts, train_tam_texts,
                                                         vocab_size=VOCAB_SIZE,
                                                         sequence_length=SEQUENCE_LENGTH)

    # Save configs and vocabs (matching original behaviour)
save_vectorizer_and_vocab(tam_vectorization, 'tam_vectorization_config.json', 'tam_vocab.json')
fre_config = fre_vectorization.get_config()
fre_config.pop('standardize', None)
with open('fre_vectorization_config.json', 'w', encoding='utf-8') as f:
    json.dump(fre_config, f)
build_and_save_fre_vocab(train_fre_texts, fre_config, 'fre_vocab.json', vocab_size=VOCAB_SIZE)

    # Prepare datasets
train_ds = make_dataset(train_pairs, fre_vectorization, tam_vectorization, batch_size=BATCH_SIZE)
val_ds = make_dataset(val_pairs, fre_vectorization, tam_vectorization, batch_size=BATCH_SIZE)

for inputs, targets in train_ds.take(1):
    print(f'inputs["encoder_inputs"].shape: {inputs["encoder_inputs"].shape}')
    print(f'inputs["decoder_inputs"].shape: {inputs["decoder_inputs"].shape}')
    print(f"targets.shape: {targets.shape}")

    # Build and train model
transformer = build_transformer(vocab_size=VOCAB_SIZE, sequence_length=SEQUENCE_LENGTH,
                                embed_dim=EMBED_DIM, latent_dim=LATENT_DIM, num_heads=NUM_HEADS)
transformer.summary()

transformer.compile(
    optimizer="rmsprop",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

transformer.fit(train_ds, epochs=EPOCHS, validation_data=val_ds)

    # Prepare decoding utilities
tam_index_lookup = build_index_lookup(tam_vectorization)

    # Test decode few random samples
test_fre_texts = [pair for pair in test_pairs['french']]
for _ in range(5):
    input_sentence = random.choice(test_fre_texts)
    input_sentence = input_sentence.lower()
    input_sentence = input_sentence.translate(str.maketrans('', '', STRIP_CHARS))
    translated = decode_sentence(input_sentence, fre_vectorization, tam_vectorization, transformer,
                                 tam_index_lookup, max_decoded_sentence_length=SEQUENCE_LENGTH)
    print(f"input: {input_sentence}")
    print(f"translated: {translated}\n")


400000 total pairs
280000 training pairs
60000 validation pairs
60000 test pairs
inputs["encoder_inputs"].shape: (128, 20)
inputs["decoder_inputs"].shape: (128, 20)
targets.shape: (128, 20)
Model: "transformer"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 encoder_inputs (InputLayer)    [(None, None)]       0           []                               
                                                                                                  
 positional_embedding (Position  (None, None, 256)   3845120     ['encoder_inputs[0][0]']         
 alEmbedding)                                                                                     
                                                                                                  
 decoder_inputs (InputLayer)    [(None, None)]       0           []                             

In [20]:
test_fre_texts = [pair for pair in test_pairs['french']]
for _ in range(5):
    input_sentence = random.choice(test_fre_texts)
    input_sentence = input_sentence.lower()
    input_sentence = input_sentence.translate(str.maketrans('', '', STRIP_CHARS))
    translated = decode_sentence(input_sentence, fre_vectorization, tam_vectorization, transformer,
                                 tam_index_lookup, max_decoded_sentence_length=SEQUENCE_LENGTH)
    print(f"input: {input_sentence}")
    print(f"translated: {translated}\n")

input: on retourne ensuite dans notre chambre avec elliot
translated: [start] எதிர் காலம் பற்றிய பல எதிர் பார்போடு எத்தனையோ கனவுகளுடன் எண்ணியே வாழ்ந்திருந்தோம் எம் ஊரில் [end]

input: et pourtant ma très chère nièce je vais te transmettre une partie de mon savoir
translated: [start] ஆமாண்டிதெரியலடிஆவாதடிஎன உனது தோழியிடம்பேசுவதுபோல நான் ஒன்றுபேசநீ ஒன்று பேசி வைத்துவிடுவாய் [end]

input: considéré comme le deuxième prix littéraire le plus prestigieux après le prix goncourt le prix renaudot offre une visibilité médiatique
translated: [start] kuycase csgo வழக்கு திறப்பு தள free daily bonus kuycase is the best place to open csவழக்குகள் go மற்றும் சிறந்த துளி

input: cest clairement une voiture orientée vers le plaisir de conduire et cela se ressent dès les premiers tours de roues
translated: [start] drive safe enjoy your motorcycling எனக் கூறி வாசல் வரை வந்து வழி அனுப்பினார் [end]

input: de manière générale une bibliothèque n’est pas un endroit où l’on ne fait que lire apprendre ou travail

In [21]:
transformer.save("french_tamil_transformer.keras")
transformer.save_weights("french_tamil_weights.weights.h5")

Model: "model_2"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 encoder_inputs (InputLayer)    [(None, None)]       0           []                               
                                                                                                  
 positional_token_embedding (Po  (None, None, 256)   3845120     ['encoder_inputs[0][0]']         
 sitionalTokenEmbedding)                                                                          
                                                                                                  
 decoder_inputs (InputLayer)    [(None, None)]       0           []                               
                                                                                                  
 transformer_encoder (Transform  (None, None, 256)   3155456     ['positional_token_embeddin

Epoch 1/3
2188/2188 [==============================] - 259s 115ms/step - loss: 0.7810 - accuracy: 0.8869 - val_loss: 0.5567 - val_accuracy: 0.9135
Epoch 2/3
2188/2188 [==============================] - 277s 127ms/step - loss: 0.5556 - accuracy: 0.9141 - val_loss: 0.5363 - val_accuracy: 0.9155
Epoch 3/3
2188/2188 [==============================] - 258s 118ms/step - loss: 0.5504 - accuracy: 0.9146 - val_loss: 0.5388 - val_accuracy: 0.9149


French: mesdames et messieurs les membres du corps diplomatique
Tamil:  [start]                    

French: j appelle tous les musulmans de france a ne pas acheter de moutons pour l aid en france pour montrer notre détermination et surtout notre puissance financières 
Tamil:  [start]                    

French: colossiens 311 autrement dit dans ce renouvellemet il ny a plus dhomme ou dhumanité mais le christ en nous soit dieu et ce qui est divin et non lhumain et lhumanité
Tamil:  [start]                    

French: or chaque année lemmanuel prend vie à nouveau dans leglise et les âmes  et pas plus aujourdhui quil y a dixhuit siècles il ne veut naître sans que vousmême ayez comme alors préparé les voies à cette nativité qui donne à chacun de nous son sauveur
Tamil:  [start]                    

French: le meilleur moment pour visiter cordoue est sans aucun doute pendant la fête des patios généralement la première semaine de mai pour contempler les cours majestueux de chaque particip

NameError: name 'tokenizer' is not defined